In [4]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 6)

# Caminho para os dados brutos
DATA_PATH = os.path.join("..", "data", "raw", "dados_sensores_aeromotion.csv")
if not os.path.exists(DATA_PATH):
    DATA_PATH = r"C:\Users\henrique_schorck\Documents\GitHub\Projeto-Integrador-Aero\Machine-Learning\Arquivos_CSV\dados_sensores_aeromotion.csv"

df_raw = pd.read_csv(DATA_PATH)
print(f"Dataset carregado com sucesso: {df_raw.shape[0]} linhas e {df_raw.shape[1]} colunas.")
df_raw.head()

SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (2285954252.py, line 13)

In [ ]:
df_inspec = df_raw.copy()
df_inspec['timestamp'] = pd.to_datetime(df_inspec['timestamp'])
df_inspec = df_inspec.sort_values(by=['id_equipamento', 'timestamp']).reset_index(drop=True)

print("--- Tipos de Dados ---")
print(df_inspec.dtypes)

print("\n--- Verificação de Frequência Temporal por Bancada (Amostragem esperada: 10 min) ---")
for eq in sorted(df_inspec['id_equipamento'].unique()):
    sub = df_inspec[df_inspec['id_equipamento'] == eq]
    deltas = sub['timestamp'].diff().value_counts(normalize=True).head(3) * 100
    print(f"\nEquipamento {eq} (Total: {len(sub)} leituras):")
    print(deltas.to_string())

print("\n--- Taxa de Ausência de Dados (NaNs em %) Agrupada por Equipamento ---")
taxa_ausentes = (df_inspec.groupby('id_equipamento').apply(lambda g: g.isnull().mean()) * 100).round(2)
taxa_ausentes = taxa_ausentes[['temperatura_mancal_c', 'vibracao_rms_mms', 'pressao_oleo_bar', 'emissao_acustica_db']]
display(taxa_ausentes)

In [ ]:
df_clean = df_inspec.copy()

qtd_outliers_temp = (df_clean['temperatura_mancal_c'] > 150.0).sum()
qtd_outliers_vib = (df_clean['vibracao_rms_mms'] < 0.0).sum()

print(f"Ruídos EMI detectados: {qtd_outliers_temp} leituras de temperatura extrema (>150°C)")
print(f"Ruídos EMI detectados: {qtd_outliers_vib} leituras de vibração negativa (<0.0 mm/s)")

# Substituição por NaN para posterior interpolação contínua
df_clean.loc[df_clean['temperatura_mancal_c'] > 150.0, 'temperatura_mancal_c'] = np.nan
df_clean.loc[df_clean['vibracao_rms_mms'] < 0.0, 'vibracao_rms_mms'] = np.nan

In [ ]:
olunas_continuas = [
    'temperatura_mancal_c', 'vibracao_rms_mms', 'pressao_oleo_bar', 
    'rotacao_rpm', 'emissao_acustica_db', 'corrente_eletrica_amp'
]

# Interpolação linear respeitando a física contínua de cada bancada isoladamente
df_interpolado = []
for eq, sub in df_clean.groupby('id_equipamento', as_index=False):
    sub = sub.sort_values('timestamp')
    sub[colunas_continuas] = sub[colunas_continuas].interpolate(method='linear')
    sub[colunas_continuas] = sub[colunas_continuas].bfill().ffill()
    df_interpolado.append(sub)

df_clean = pd.concat(df_interpolado, ignore_index=True)
print("Valores ausentes restantes após saneamento:")
print(df_clean.isnull().sum())

In [ ]:
f_features_list = []

for eq, sub in df_clean.groupby('id_equipamento', as_index=False):
    sub = sub.sort_values('timestamp').set_index('timestamp')
    
    # Janela de 1 hora
    sub['vibracao_mean_1h'] = sub['vibracao_rms_mms'].rolling('1h', min_periods=1).mean()
    sub['vibracao_std_1h'] = sub['vibracao_rms_mms'].rolling('1h', min_periods=1).std().fillna(0)
    sub['temp_mean_1h'] = sub['temperatura_mancal_c'].rolling('1h', min_periods=1).mean()
    sub['temp_std_1h'] = sub['temperatura_mancal_c'].rolling('1h', min_periods=1).std().fillna(0)
    
    # Janela de 6 horas
    sub['vibracao_mean_6h'] = sub['vibracao_rms_mms'].rolling('6h', min_periods=1).mean()
    sub['vibracao_std_6h'] = sub['vibracao_rms_mms'].rolling('6h', min_periods=1).std().fillna(0)
    sub['temp_mean_6h'] = sub['temperatura_mancal_c'].rolling('6h', min_periods=1).mean()
    sub['temp_std_6h'] = sub['temperatura_mancal_c'].rolling('6h', min_periods=1).std().fillna(0)
    
    # Feature Derivada: Severidade Dinâmica (Vibração RMS / Rotação RPM)
    sub['severidade_dinamica'] = sub['vibracao_rms_mms'] / sub['rotacao_rpm']
    
    sub = sub.reset_index()
    df_features_list.append(sub)

df_final = pd.concat(df_features_list, ignore_index=True)
df_final = df_final.sort_values(by=['timestamp', 'id_equipamento']).reset_index(drop=True)
print(f"Features calculadas com sucesso! Formato final: {df_final.shape}")
df_final[['timestamp', 'id_equipamento', 'vibracao_mean_1h', 'severidade_dinamica']].head()

In [ ]:
features_analise = [
    'temperatura_mancal_c', 'vibracao_rms_mms', 'pressao_oleo_bar', 
    'rotacao_rpm', 'emissao_acustica_db', 'corrente_eletrica_amp',
    'severidade_dinamica', 'vibracao_mean_1h', 'temp_mean_1h', 'tipo_falha'
]

plt.figure(figsize=(10, 8))
corr = df_final[features_analise].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, cbar=True)
plt.title("Matriz de Correlação Cruzada das Variáveis Operacionais e Alvo")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.boxplot(data=df_final, x='modo_operacao', y='rotacao_rpm', ax=axes[0], palette="Blues_d")
axes[0].set_title("Rotação (RPM) por Regime de Carga")

sns.boxplot(data=df_final, x='modo_operacao', y='temperatura_mancal_c', ax=axes[1], palette="Reds_d")
axes[1].set_title("Temperatura do Mancal (°C) por Regime de Carga")

sns.boxplot(data=df_final, x='modo_operacao', y='vibracao_rms_mms', ax=axes[2], palette="Greens_d")
axes[2].set_title("Vibração RMS (mm/s) por Regime de Carga")
plt.tight_layout()
plt.show()

In [ ]:
PROCESSED_DIR = os.path.join("..", "data", "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)
OUTPUT_FILE = os.path.join(PROCESSED_DIR, "dados_sensores_tratados.csv")
df_final.to_csv(OUTPUT_FILE, index=False)
print(f"Dataset exportado com sucesso para: {OUTPUT_FILE}")